# 05 — Edge Resilience, Outage State Machine & Closed F5 Diagnostic Catalog

Specification v0.3.0 defines a **closed catalog of 77 machine-readable codes** (`internal/core/codes/catalog.tsv`). This notebook tests key security, lifecycle, rule-validation, and outage-resilience invariants live:

1. **Authentication & Stewardship Guards (`C-HUB`)**: Expired JWTs (`401 API_UNAUTHENTICATED`), wrong audience, invalid slugs (`422 NS_SLUG_INVALID`), duplicate namespaces (`409 NS_EXISTS`), and vocabulary kind/media mismatches.
2. **Lifecycle Date & Revocation Invariants (`§3.3.2`, `DATE-1`/`DATE-2`, `REV-1`..`REV-4`)**: `DATES_GRACE_ON_COMPATIBLE`, `DATES_NON_MONOTONIC`, `DATES_TOO_MANY_VERSIONS`, `DATES_REVOKE_LEAVES_NO_HEAD`, `DATES_JUSTIFICATION_REQUIRED`, and `DATES_REVOCATION_TOO_EARLY`.
3. **Rule Engine Validation (`C-CLI`)**: `RULES_SCHEMA`, `RULES_DUPLICATE_ID`, `RULES_DUPLICATE_PRIORITY`, and `RULES_GATE_MESSAGE_MISMATCH`.
4. **Hub Outage & Operational State Transitions (`Active` → `Degraded` → `FailClosed`)**: Stopping `alexandria-server` and advancing the clock past `ttl` and `ttl + max_staleness` (`CLIENT_FAIL_CLOSED`).

In [ ]:
from lab import AlexandriaLab, show_events_table, show_state_machine

lab = AlexandriaLab(
    namespace="demo",
    compiler="poc",
    initial_clock="2026-09-01T10:00:00Z",
    lifecycle={
        "ttl": "PT10M",
        "sync_interval": "PT2M",
        "max_staleness": "PT20M",
        "propagation_window": "PT5M",
        "min_grace_window": "PT10M",
        "grace_window": "PT30M",
        "max_concurrent_versions": 2,
        "clock_skew": "PT5S",
        "lease_margin": "PT10S",
        "degraded": {"policy_intake": "continue"},
    },
)
lab.start_hub_stack()
ana = lab.token("ana", admin=True)
sergio = lab.token("sergio")

## 1. Hub Authentication, Namespace & Vocabulary Negative Checks

In [ ]:
# Expired token (issued 2 hours in the past with 1h TTL) -> 401 API_UNAUTHENTICATED
expired_tok = lab.token("ana", admin=True, ttl="1h", clock_override="2026-09-01T07:00:00Z")
st, _, err_exp = lab.request("POST", f"{lab.hub_url}/v1/namespaces", token=expired_tok, body={"slug": "demo"})
assert st == 401 and err_exp["code"] == "API_UNAUTHENTICATED"

# Wrong audience -> 401 API_UNAUTHENTICATED
wrong_aud_tok = lab.token("ana", admin=True, aud="other-service")
st, _, err_aud = lab.request("POST", f"{lab.hub_url}/v1/namespaces", token=wrong_aud_tok, body={"slug": "demo"}, verbose=False)
assert st == 401 and err_aud["code"] == "API_UNAUTHENTICATED"

# Invalid namespace slug ("Demo_1") & reserved "_vhub"
st, _, err_slug = lab.request("POST", f"{lab.hub_url}/v1/namespaces", token=ana, body={"slug": "Demo_1"})
assert st == 422 and err_slug["code"] == "NS_SLUG_INVALID"

# Create 'demo', check duplicate 409 NS_EXISTS, assign 'sergio'
lab.request("POST", f"{lab.hub_url}/v1/namespaces", token=ana, body={"slug": "demo"}, verbose=False)
st, _, err_dup = lab.request("POST", f"{lab.hub_url}/v1/namespaces", token=ana, body={"slug": "demo"})
assert st == 409 and err_dup["code"] == "NS_EXISTS"

lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/stewards", token=ana, body={"iss": lab.idp_url, "sub": "sergio"}, verbose=False)

# Hosted vocabulary kind/media-type checks
lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/vocabularies", token=sergio, body={"name": "ont", "kind": "hosted", "media_type": "application/ld+json"}, verbose=False)
st, _, err_kind = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/vocabularies/ont:refresh", token=sergio)
assert st == 409 and err_kind["code"] == "VOCAB_KIND_MISMATCH"

st, _, err_mt = lab.request("PUT", f"{lab.hub_url}/v1/namespaces/demo/vocabularies/ont/content", token=sergio, headers={"Content-Type": "text/turtle"}, body="<a> <b> <c> .")
assert st == 415 and err_mt["code"] == "VOCAB_MEDIA_TYPE_MISMATCH"

## 2. Lifecycle Date Planning (`§3.3.2`, `DATE-1`, `DATE-2`) & Revocation Guards (`REV-1`..`REV-4`)

In [ ]:
def make_cand(version: str) -> str:
    body = {
        "version": version,
        "profiles": [f"{lab.hub_url}/pub/demo/profiles/odrl/{version}"],
        "slots": [{"slot": "policy", "obligation": "mandatory", "applies_to": ["policy_intake"]}],
        "files": [
            {
                "path": "schemas/policy.schema.json",
                "media_type": "application/schema+json",
                "role": "json-schema",
                "slot": "policy",
                "applies_to": ["policy_intake"],
                "content": {"type": "object", "properties": {"action": {"enum": ["odrl:use"]}}, "required": ["action"]},
            }
        ],
    }
    _, hd, _ = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/drafts", token=sergio, body=body, verbose=False)
    did = hd["location"].rsplit("/", 1)[-1]
    _, _, c = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/drafts/{did}:propose", token=sergio, verbose=False)
    return c["id"].rsplit("/", 1)[-1]

# Approve 1.0.0 (initial)
c100 = make_cand("1.0.0")
st, _, _ = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/candidates/{c100}:approve", token=sergio, verbose=False)
assert st == 201

# REV-1..REV-4 checks on 1.0.0 while it is the sole head version
st, _, e_just = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/versions/1.0.0:revoke", token=sergio, body={"justification": "   "})
assert st == 422 and e_just["code"] == "DATES_JUSTIFICATION_REQUIRED"

st, _, e_early = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/versions/1.0.0:revoke", token=sergio, body={"justification": "test", "revoked_from": "2026-09-01T10:01:00Z"})
assert st == 422 and e_early["code"] == "DATES_REVOCATION_TOO_EARLY"

st, _, e_nohead = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/versions/1.0.0:revoke", token=sergio, body={"justification": "sole version"})
assert st == 422 and e_nohead["code"] == "DATES_REVOKE_LEAVES_NO_HEAD"

# Compatible version 1.1.0 cannot specify grace_window -> DATES_GRACE_ON_COMPATIBLE
c110 = make_cand("1.1.0")
st, _, e_gcomp = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/candidates/{c110}:approve", token=sergio, body={"grace_window": "PT15M"})
assert st == 422 and e_gcomp["code"] == "DATES_GRACE_ON_COMPATIBLE"

# Approve breaking 2.0.0 (enforced_from = 10:00 + 5m + 30m = 10:35)
c200 = make_cand("2.0.0")
st, _, _ = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/candidates/{c200}:approve", token=sergio, verbose=False)
assert st == 201

# DATE-1: 3.0.0 with shorter PT10M grace would enforce at 10:15 < 10:35 -> DATES_NON_MONOTONIC
c300 = make_cand("3.0.0")
st, _, e_mono = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/candidates/{c300}:approve", token=sergio, body={"grace_window": "PT10M"})
assert st == 422 and e_mono["code"] == "DATES_NON_MONOTONIC"

# DATE-2: At 10:06 (1.0.0 and 2.0.0 both in force), approving 2.1.0 would put 3 versions in force (> max_concurrent_versions=2) -> DATES_TOO_MANY_VERSIONS
lab.set_clock("2026-09-01T10:06:00Z", silent=True)
c210 = make_cand("2.1.0")
st, _, e_many = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/candidates/{c210}:approve", token=sergio)
assert st == 422 and e_many["code"] == "DATES_TOO_MANY_VERSIONS"

## 3. Client Admin Rule Validation & Hub Outage (`Active` → `Degraded` → `FailClosed`)

Let's advance the clock past `2.0.0`'s enforcement (`2026-09-01T10:40:00Z`), start `alexandria-client`, verify rule-validation errors (`RULES_SCHEMA`, `RULES_DUPLICATE_ID`, `RULES_DUPLICATE_PRIORITY`, `RULES_GATE_MESSAGE_MISMATCH`), and then **stop `alexandria-server`** to watch the client transition cleanly from `Active` → `Degraded` → `FailClosed`!

In [ ]:
try:
    lab.set_clock("2026-09-01T10:40:00Z")
    # Re-issue a fresh listing at 10:40 by revoking retired 1.0.0 cleanly (2.0.0 is now head!)
    st, _, _ = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/versions/1.0.0:revoke", token=sergio, body={"justification": "1.0.0 superseded by 2.0.0"}, verbose=False)
    assert st == 200

    lab.start_client()
    lab.wait_synced("2026-09-01T10:40:00Z")
    _, hr_rules, _ = lab.request("GET", f"{lab.admin_url}/rules", verbose=False)
    etag = hr_rules["etag"]

    # 1. Rule engine negative validations
    st, _, r_schema = lab.request("POST", f"{lab.admin_url}/rules", headers={"If-Match": etag}, body={"id": "bad"})
    assert st == 422 and r_schema["code"] == "RULES_SCHEMA"

    st, _, r_dupid = lab.request(
        "POST", f"{lab.admin_url}/rules", headers={"If-Match": etag},
        body={"id": "contract-request", "priority": 999, "listener": "dsp-ingress", "match": {"method": "POST", "path": "/x"}, "message": "ContractRequestMessage", "verify": {"gates": ["dsp_message"]}, "mode": "observe"},
    )
    assert st == 422 and r_dupid["code"] == "RULES_DUPLICATE_ID"

    st, _, r_dupprio = lab.request(
        "POST", f"{lab.admin_url}/rules", headers={"If-Match": etag},
        body={"id": "dup-prio", "priority": 100, "listener": "dsp-ingress", "match": {"method": "POST", "path": "/x"}, "message": "ContractRequestMessage", "verify": {"gates": ["dsp_message"]}, "mode": "observe"},
    )
    assert st == 422 and r_dupprio["code"] == "RULES_DUPLICATE_PRIORITY"

    st, _, r_gate = lab.request(
        "POST", f"{lab.admin_url}/rules", headers={"If-Match": etag},
        body={"id": "bad-gate", "priority": 999, "listener": "dsp-ingress", "match": {"method": "POST", "path": "/x"}, "message": "ContractRequestMessage", "verify": {"gates": ["agreement_admission"]}, "mode": "observe"},
    )
    assert st == 422 and r_gate["code"] == "RULES_GATE_MESSAGE_MISMATCH"

    # 2. Stop alexandria-server! Listing valid_from=10:40, ttl=PT10M (valid_until=10:50), max_staleness=PT20M (fail_closed=11:10)
    lab.stop_server()
    msg_ok = {"@type": "ContractRequestMessage", "providerPid": "urn:uuid:p-1", "consumerPid": "urn:uuid:c-1", "action": "odrl:use"}

    # At 10:55 (> valid_until 10:50, < 11:10) -> Degraded! Readyz is still 200 and degraded.policy_intake=continue forwards traffic
    lab.set_clock("2026-09-01T10:55:00Z")
    _, _, st_deg = lab.request("GET", f"{lab.admin_url}/status")
    assert st_deg["state"] == "Degraded"
    show_state_machine(st_deg["state"], subtitle="Hub unreachable; operating within max_staleness (10:50..11:10)")

    st_rdz_deg, _, _ = lab.request("GET", f"{lab.admin_url}/readyz", verbose=False)
    assert st_rdz_deg == 200
    st_msg_deg, _, _ = lab.request("POST", f"{lab.dsp_ingress_url}/negotiations/urn:uuid:p-1/request", body=msg_ok)
    assert st_msg_deg == 200

    # At 11:15 (> valid_until + max_staleness = 11:10) -> FailClosed! Readyz returns 503 and enforce route rejects with CLIENT_FAIL_CLOSED
    lab.set_clock("2026-09-01T11:15:00Z")
    _, _, st_fc = lab.request("GET", f"{lab.admin_url}/status")
    assert st_fc["state"] == "FailClosed"
    show_state_machine(st_fc["state"], subtitle="max_staleness exceeded -> FailClosed (enforce routes reject)")

    st_rdz_fc, _, _ = lab.request("GET", f"{lab.admin_url}/readyz")
    assert st_rdz_fc == 503

    st_msg_fc, hdr_fc, err_fc = lab.request("POST", f"{lab.dsp_ingress_url}/negotiations/urn:uuid:p-1/request", body=msg_ok)
    assert st_msg_fc == 400 and err_fc["code"] == "CLIENT_FAIL_CLOSED"
    assert 'code="CLIENT_FAIL_CLOSED"' in hdr_fc.get("vhub-verdict", "")

    show_events_table(lab.read_events(), title="Client State & Resilience Events")
finally:
    lab.close()